In [2]:
import os

import numpy as np
import prettytable as pt
import pandas as pd
from scipy.optimize import curve_fit
from scipy.integrate import quad

import hist

In [3]:
cats = {
    # 'VBFHH': {
    #     "is_VBFHH_sig": 0.774,
    # },
    'cat1': {
        "is_ggHH_sig": 0.20105975753991212,
        "is_nonRes_bkg": 0.00033920225777953755,
        "is_Res_bkg": 0.5155782175840005
    },
    'cat2': {
        "is_ggHH_sig": 0.8633095100474404,
        "is_nonRes_bkg": 0.0015707913189143552,
        "is_Res_bkg": 0.7641090808766726
    },
    'cat3': {
        "is_ggHH_sig": 0.7121864823883823,
        "is_nonRes_bkg": 0.006492482419944794,
        "is_Res_bkg": 0.5593729774038129
    }
}
# root://redirector.t2.ucsd.edu:1095//store/user/evourlio/Run3HHbbgg_multiclassDNN/storeForFilesFromCorrespondingUAFDirs/HHbbgg_conditional_classifiers_afterPAS/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs
dirpath = "/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs"
samples = os.listdir(dirpath)
files = [os.path.join(dirpath, sample, "scored_events.parquet") for sample in samples]
files

['/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/X_train.npy/scored_events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/X_val.npy/scored_events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/after_random_search_best1/scored_events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/class_weights_for_training_abs.npy/scored_events.parquet',
 '/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/class_weights_for_val

In [4]:
print("/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/individual_samples/2024/GluGlutoHHto2B2G_kl_1p00_kt_1p00_c2_0p00/scored_events.parquet")
signal = pd.read_parquet("/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/individual_samples/2024/GluGlutoHHto2B2G_kl_1p00_kt_1p00_c2_0p00/scored_events.parquet")

/eos/uscms/store/group/lpcdihiggsboost/tsievert/Version_20260222_bTagWPs_revisedNewBaseline_ResNonResggHHVBFHH_xsecWeightInSignal_1D_Run2Run3_stopAt140Epochs/individual_samples/2024/GluGlutoHHto2B2G_kl_1p00_kt_1p00_c2_0p00/scored_events.parquet


In [5]:
input_vars = [
    "mass", "lumi", "event", 
    
    "eta", 
    
    "lead_eta", "lead_phi", "lead_mvaID", 
    "sublead_eta", "sublead_phi", "sublead_mvaID", 
    
    "nonResReg_vbfpair_lead_bjet_eta", "nonResReg_vbfpair_lead_bjet_phi", 
    "nonResReg_vbfpair_lead_bjet_btag_WP_L", "nonResReg_vbfpair_lead_bjet_btag_WP_M", 
    "nonResReg_vbfpair_lead_bjet_btag_WP_T", "nonResReg_vbfpair_lead_bjet_btag_WP_XT", "nonResReg_vbfpair_lead_bjet_btag_WP_XXT", 
    
    "nonResReg_vbfpair_sublead_bjet_eta", "nonResReg_vbfpair_sublead_bjet_phi", 
    "nonResReg_vbfpair_sublead_bjet_btag_WP_L", "nonResReg_vbfpair_sublead_bjet_btag_WP_M", 
    "nonResReg_vbfpair_sublead_bjet_btag_WP_T", "nonResReg_vbfpair_sublead_bjet_btag_WP_XT", "nonResReg_vbfpair_sublead_bjet_btag_WP_XXT", 
    
    "nonResReg_vbfpair_DeltaR_j1g1", "nonResReg_vbfpair_DeltaR_j2g1", 
    "nonResReg_vbfpair_DeltaR_j1g2", "nonResReg_vbfpair_DeltaR_j2g2", "nonResReg_vbfpair_DeltaR_jg_min", 
    
    "nonResReg_vbfpair_CosThetaStar_CS", "nonResReg_vbfpair_CosThetaStar_gg", "nonResReg_vbfpair_CosThetaStar_jj", 
    
    "puppiMET_phi", "puppiMET_pt", 
    
    "n_leptons", "n_jets", 
    
    "nonResReg_vbfpair_chi_t0", "nonResReg_vbfpair_chi_t1", 
    
    "nonResReg_vbfpair_DeltaPhi_j1MET", "nonResReg_vbfpair_DeltaPhi_j2MET", 
    
    "nonResReg_vbfpair_VBF_first_jet_eta", "nonResReg_vbfpair_VBF_first_jet_phi", 
    "nonResReg_vbfpair_VBF_second_jet_eta", "nonResReg_vbfpair_VBF_second_jet_phi", 
    "nonResReg_vbfpair_VBF_first_jet_PtOverM", "nonResReg_vbfpair_VBF_second_jet_PtOverM", 
    "nonResReg_vbfpair_VBF_jet_eta_prod", "nonResReg_vbfpair_VBF_jet_eta_diff", 
    "nonResReg_vbfpair_VBF_jet_eta_sum", "nonResReg_vbfpair_VBF_DeltaR_j1b1", 
    "nonResReg_vbfpair_VBF_DeltaR_j1b2", "nonResReg_vbfpair_VBF_DeltaR_j2b1", 
    "nonResReg_vbfpair_VBF_DeltaR_j2b2", "nonResReg_vbfpair_VBF_DeltaR_j1g1", 
    "nonResReg_vbfpair_VBF_DeltaR_j1g2", "nonResReg_vbfpair_VBF_DeltaR_j2g1", 
    "nonResReg_vbfpair_VBF_DeltaR_j2g2", "nonResReg_vbfpair_VBF_DeltaR_jb_min", 
    "nonResReg_vbfpair_VBF_DeltaR_jg_min", "nonResReg_vbfpair_VBF_dijet_mass",
     
    "nonResReg_vbfpair_HHbbggCandidate_eta", 
    "diphoton_PtOverM_ggjj", 
    "nonResReg_vbfpair_dijet_PtOverM_ggjj", 
    "nonResReg_vbfpair_lead_bjet_over_M_regressed", 
    "nonResReg_vbfpair_sublead_bjet_over_M_regressed", 
    "nonResReg_vbfpair_dijet_mass_DNNreg"
]
input_signal = signal.loc[:, ["weight_tot", "is_nonRes_bkg_score", "is_Res_bkg_score", "is_ggHH_sig_score", "is_VBFHH_sig_score"]+input_vars]

In [6]:
input_signal.head()

,weight_tot,is_nonRes_bkg_score,is_Res_bkg_score,is_ggHH_sig_score,is_VBFHH_sig_score,mass,lumi,event,eta,lead_eta,...,nonResReg_vbfpair_VBF_DeltaR_j2g2,nonResReg_vbfpair_VBF_DeltaR_jb_min,nonResReg_vbfpair_VBF_DeltaR_jg_min,nonResReg_vbfpair_VBF_dijet_mass,nonResReg_vbfpair_HHbbggCandidate_eta,diphoton_PtOverM_ggjj,nonResReg_vbfpair_dijet_PtOverM_ggjj,nonResReg_vbfpair_lead_bjet_over_M_regressed,nonResReg_vbfpair_sublead_bjet_over_M_regressed,nonResReg_vbfpair_dijet_mass_DNNreg
0,0.000032,0.004912,0.026198,0.956464,0.012426,127.370724,153,7148,0.848661,1.108154,...,-999.0,-999.0,-999.0,-999.0,0.473444,0.301582,0.321593,1.221259,0.227124,111.311974
1,-0.000031,0.000131,0.001346,0.994592,0.003931,120.835401,153,7151,1.304075,1.651367,...,-999.0,-999.0,-999.0,-999.0,3.714351,0.337643,0.448333,1.119640,0.857878,138.783708
2,0.000011,0.000448,0.003670,0.993217,0.002664,123.036715,153,7154,-2.263550,-2.253906,...,-999.0,-999.0,-999.0,-999.0,-4.961024,0.373796,0.352247,1.023193,0.548629,111.180941
3,0.000032,0.000798,0.015221,0.977263,0.006718,125.005579,153,7153,0.161119,0.266602,...,-999.0,-999.0,-999.0,-999.0,-0.281693,0.502191,0.267490,0.981649,0.242821,115.465395
4,0.000041,0.068456,0.153095,0.763173,0.015276,124.300453,153,7145,0.815960,0.846924,...,-999.0,-999.0,-999.0,-999.0,0.819691,0.520338,0.214054,0.680613,0.595847,71.612961


In [8]:
preproc_file = "/eos/uscms/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/preprocessed/v7/Run3_2024/sim/GluGlutoHH_kl-1p00_kt-1p00_c2-0p00/NOTAG_preprocessed.parquet"
preproc_signal = pd.read_parquet(preproc_file)

In [28]:
preproc_input_signal = preproc_signal.loc[:, ["eventWeight"]+[var.replace('btag_WP_', 'bTagWP') for var in input_vars]]

In [29]:
preproc_input_signal.head()

,eventWeight,mass,lumi,event,eta,lead_eta,lead_phi,lead_mvaID,sublead_eta,sublead_phi,...,nonResReg_vbfpair_VBF_DeltaR_j2g2,nonResReg_vbfpair_VBF_DeltaR_jb_min,nonResReg_vbfpair_VBF_DeltaR_jg_min,nonResReg_vbfpair_VBF_dijet_mass,nonResReg_vbfpair_HHbbggCandidate_eta,diphoton_PtOverM_ggjj,nonResReg_vbfpair_dijet_PtOverM_ggjj,nonResReg_vbfpair_lead_bjet_over_M_regressed,nonResReg_vbfpair_sublead_bjet_over_M_regressed,nonResReg_vbfpair_dijet_mass_DNNreg
0,0.000032,127.370724,153,7148,0.848661,1.108154,-0.720947,0.772336,-0.086243,-1.927734,...,-999.0,-999.0,-999.0,-999.0,0.473444,0.301582,0.321593,1.221259,0.227124,111.311974
1,-0.000031,120.835401,153,7151,1.304075,1.651367,0.234528,0.921776,0.495422,-0.307190,...,-999.0,-999.0,-999.0,-999.0,3.714351,0.337643,0.448333,1.119640,0.857878,138.783708
2,0.000011,123.036715,153,7154,-2.263550,-2.253906,1.472168,0.802558,-1.444092,2.763672,...,-999.0,-999.0,-999.0,-999.0,-4.961024,0.373796,0.352247,1.023193,0.548629,111.180941
3,0.000032,125.005579,153,7153,0.161119,0.266602,3.072754,0.790100,-0.156982,-1.945557,...,-999.0,-999.0,-999.0,-999.0,-0.281693,0.502191,0.267490,0.981649,0.242821,115.465395
4,0.000040,124.300453,153,7145,0.815960,0.846924,-0.250549,0.081905,0.202087,-1.464844,...,-999.0,-999.0,-999.0,-999.0,0.819691,0.520338,0.214054,0.680613,0.595847,71.612961


In [35]:
print(f"Num SnT signal events: {len(input_signal)}")
print(f"Num our preprocessed signal events[ 100 < diphoton mass < 180 ]: {len(input_signal.loc[np.logical_and(input_signal['mass'].gt(100).to_numpy(), input_signal['mass'].lt(180).to_numpy())])}")
print(f"Num our preprocessed signal events[ lead and sublead photons MVAID > -0.7 ]: {len(input_signal.loc[np.logical_and(input_signal['lead_mvaID'].gt(-0.7).to_numpy(), input_signal['sublead_mvaID'].gt(-0.7).to_numpy())])}")
print('-'*60)
print(f"Num our preprocessed signal events: {len(preproc_input_signal)}")
preproc_dipho_mass_window = np.logical_and(preproc_input_signal['mass'].gt(100).to_numpy(), preproc_input_signal['mass'].lt(180).to_numpy())
print(f"Num our preprocessed signal events[ 100 < diphoton mass < 180 ]: {len(preproc_input_signal.loc[preproc_dipho_mass_window])}")
phoMVA_cuts = np.logical_and(preproc_input_signal['lead_mvaID'].gt(-0.7).to_numpy(), preproc_input_signal['sublead_mvaID'].gt(-0.7).to_numpy())
print(f"Num our preprocessed signal events[ lead and sublead photons MVAID > -0.7 ]: {len(preproc_input_signal.loc[phoMVA_cuts])}")
SnT_cuts = np.logical_and(preproc_dipho_mass_window, phoMVA_cuts)
print(f"Num our preprocessed signal events[ lead and sublead photons MVAID > -0.7 ]: {len(preproc_input_signal.loc[SnT_cuts])}")

Num SnT signal events: 165115
Num our preprocessed signal events[ 100 < diphoton mass < 180 ]: 165115
Num our preprocessed signal events[ lead and sublead photons MVAID > -0.7 ]: 165115
------------------------------------------------------------
Num our preprocessed signal events: 168710
Num our preprocessed signal events[ 100 < diphoton mass < 180 ]: 167867
Num our preprocessed signal events[ lead and sublead photons MVAID > -0.7 ]: 165736
Num our preprocessed signal events[ lead and sublead photons MVAID > -0.7 ]: 165115


In [37]:
print(f"Sum SnT signal: {input_signal.loc[:, 'weight_tot'].sum()}")
print(f"Sum our preprocessed signal: {preproc_input_signal.loc[:, 'eventWeight'].sum()}")
print(f"Sum our preprocessed signal w/ SnT cuts: {preproc_input_signal.loc[SnT_cuts, 'eventWeight'].sum()}")

Sum SnT signal: 3.645386497750374
Sum our preprocessed signal: 3.716823359342285
Sum our preprocessed signal w/ SnT cuts: 3.636755392438306


In [22]:
for preproc_col, snt_col in zip(preproc_input_signal.columns, input_signal.columns):
    print()
    print(f"Checking our preprocessed column \'{preproc_col}\', against SnT's column \'{snt_col}\'")
    print(preproc_input_signal.loc[:, preproc_col].head())
    print('-'*60)
    print(input_signal.loc[:, snt_col].head())
    # print(f"Whole columns equal? {np.all(preproc_input_signal.loc[:, preproc_col].to_numpy() == input_signal.loc[:, snt_col].to_numpy())}")



Checking our preprocessed column 'eventWeight', against SnT's column 'weight_tot'
0    0.000032
1   -0.000031
2    0.000011
3    0.000032
4    0.000040
Name: eventWeight, dtype: float64
------------------------------------------------------------
0    0.000032
1   -0.000031
2    0.000011
3    0.000032
4    0.000041
Name: weight_tot, dtype: float64

Checking our preprocessed column 'lumi', against SnT's column 'lumi'
0    153
1    153
2    153
3    153
4    153
Name: lumi, dtype: uint32
------------------------------------------------------------
0    153
1    153
2    153
3    153
4    153
Name: lumi, dtype: uint32

Checking our preprocessed column 'event', against SnT's column 'event'
0    7148
1    7151
2    7154
3    7153
4    7145
Name: event, dtype: uint64
------------------------------------------------------------
0    7148
1    7151
2    7154
3    7153
4    7145
Name: event, dtype: uint64

Checking our preprocessed column 'eta', against SnT's column 'eta'
0    0.848661
1    1.

In [3]:
test = pd.read_parquet(files[0])
test.columns

Index(['sample', 'year', 'DeltaR_b1l1', 'DeltaR_b1l2', 'DeltaR_b2l1',
       'DeltaR_b2l2', 'DeltaR_j10l1', 'DeltaR_j10l2', 'DeltaR_j10l3',
       'DeltaR_j10l4',
       ...
       'nonResReg_vbfpair_sublead_bjet_btag_WP_T',
       'nonResReg_vbfpair_sublead_bjet_btag_WP_XT',
       'nonResReg_vbfpair_sublead_bjet_btag_WP_XXT', 'rel_xsec_weight',
       'weight_tot', 'score', 'is_nonRes_bkg_score', 'is_Res_bkg_score',
       'is_ggHH_sig_score', 'is_VBFHH_sig_score'],
      dtype='str', length=1774)

In [4]:
pd.unique(test['year'])

array([2024])

In [6]:
#############################################################
# ASCii histogram for rapid plotting
def ascii_hist(x, bins=10, weights=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max(N.max())
    for (xi, n) in zip(X,N):
        bar = '#'*int(n*width/nmax)
        xi = '{0: <8.4g}'.format(xi).ljust(10)
        print('{0}| {1}'.format(xi,bar))
def ascii_hist(x, bins=10, weights=None, fit=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max([N.max(), fit.max()])
    if fit is None:
        for (xi, n) in zip(X,N):
            bar = '#'*int(n*width/nmax)
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))
    else:
        for (xi, n, fiti) in zip(X,N,fit):
            bar = '#'*int(n*width/nmax)
            if fiti > n: bar = bar + ' '*(int(fiti*width/nmax) - int(n*width/nmax)) + '+'
            else: bar = ''.join([bar[j] if j != int(fiti*width/nmax) else '+' for j in range(len(bar))])
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))

#############################################################
# Sideband fit functions for nonRes bkg estimaton
def exp_func(x, a, b):
    return a * np.exp(b * x)
def sd_hist(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float]):
    return hist.Hist(
        hist.axis.Regular(int((fit_bins[1]-fit_bins[0])//fit_bins[2]), fit_bins[0], fit_bins[1], name="var", growth=False, underflow=False, overflow=False), 
        storage='weight'
    ).fill(var=mass, weight=weight)
def exp_mass_fit(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sigma: bool=False):
    _hist_ = sd_hist(mass, weight, fit_bins)
    params, _ = curve_fit(
        exp_func, _hist_.axes.centers[0]-_hist_.axes.centers[0][0], _hist_.values(), p0=(_hist_.values()[0], -0.1), 
        sigma=np.where(_hist_.values() != 0, np.sqrt(_hist_.variances()), 0.76) if sigma else None
    )
    print(_hist_)
    ascii_hist(mass, bins=np.arange(fit_bins[0], fit_bins[1], fit_bins[2]), weights=weight, fit=exp_func(_hist_.axes.centers[0]-_hist_.axes.centers[0][0], a=params[0], b=params[1]))
    return _hist_, params
def est_yield(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sr_masscut: list[float], sigma: bool=False):
    _hist_, fit_params = exp_mass_fit(mass, weight, fit_bins, sigma=sigma)
    return quad(exp_func, sr_masscut[0]-_hist_.axes.centers[0][0], sr_masscut[1]-_hist_.axes.centers[0][0], args=tuple(fit_params))[0] / fit_bins[2]

In [11]:
samples_no_vbfhh = samples[:6] + samples[7:]
print(samples_no_vbfhh)
field_names = ['Category'] + samples_no_vbfhh + ['nonRes SB fit', 's/b', 's/b with fit']
table = pt.PrettyTable(field_names=field_names, float_format=".3")

not_prev_cut_mask = {}
for name, cuts in cats.items():
    new_row = [name]
    nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

    for sample in samples_no_vbfhh:
        sample_sum = 0.

        for file in files:
            if sample not in file: continue

            file_df = pd.read_parquet(file, columns=['mass', 'weight_tot', 'is_nonRes_bkg_score', 'is_Res_bkg_score', 'is_ggHH_sig_score', 'is_VBFHH_sig_score'])
                
            if file not in not_prev_cut_mask: not_prev_cut_mask[file] = np.ones(len(file_df), dtype=bool)
            
            pass_cut_mask = not_prev_cut_mask[file]
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, file_df.loc[:, f'{cut_name}_score'].gt(cut).to_numpy() if cut_name.endswith('sig') else file_df.loc[:, f'{cut_name}_score'].lt(cut).to_numpy()
                )
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(file_df.loc[:, 'mass'].gt(122.5).to_numpy(), file_df.loc[:, 'mass'].lt(127.).to_numpy())
            )

            sample_sum += file_df.loc[pass_cut_sr_mask, 'weight_tot'].sum()

            if sample in ['DDQCDGJET', 'GGJets', 'TTGG']:
                pass_cut_sideband_mask = np.logical_and(
                    pass_cut_mask,
                    np.logical_or(file_df.loc[:, 'mass'].lt(120.).to_numpy(), file_df.loc[:, 'mass'].gt(130.).to_numpy())
                )
                nonRes_sideband = pd.concat([nonRes_sideband, file_df.loc[pass_cut_sideband_mask, ['mass', 'weight_tot']]], ignore_index=True)

            not_prev_cut_mask[file] = np.logical_and(not_prev_cut_mask[file], ~pass_cut_mask)

        new_row.append(sample_sum)

    sb_est_yield = est_yield(nonRes_sideband['mass'], nonRes_sideband['weight_tot'], [100., 180., 5.], [122.5, 127.])
    new_row.append(sb_est_yield)

    print(f"singleH samples: {[field_names[1], field_names[4]]+ field_names[7:10]}")
    print(f"nonres samples: {[field_names[2], field_names[3], field_names[6]]}")
    sum_singleH = new_row[1] + new_row[4] + sum(new_row[7:10])
    sum_bkg_no_fit = new_row[2] + new_row[3] + new_row[6]
    new_row.append(new_row[5] / (sum_singleH + sum_bkg_no_fit))
    new_row.append(new_row[5] / (sum_singleH + sb_est_yield))

    table.add_row(new_row)

print(table)


['BBHto2G_M_125', 'DDQCDGJET', 'GGJets', 'GluGluHToGG_M_125', 'GluGlutoHHto2B2G_kl_1p00_kt_1p00_c2_0p00', 'TTGG', 'VBFHToGG_M_125', 'VHtoGG_M_125', 'ttHtoGG_M_125']
                  ┌──────────────────────────────────────────────────────────┐
[100, 105) 3.538  │█████████████████████████████████████████████████████████ │
[105, 110) 1.256  │████████████████████▎                                     │
[110, 115) 0.5933 │█████████▌                                                │
[115, 120) 0.9952 │████████████████                                          │
[120, 125) 0      │                                                          │
[125, 130) 0      │                                                          │
[130, 135) 0.5042 │████████▏                                                 │
[135, 140) 0.5745 │█████████▎                                                │
[140, 145) 0.3614 │█████▉                                                    │
[145, 150) 0.3507 │█████▋                    